# Pregunta 2 — Mapeo de inundaciones con Sentinel-1 (U-Net)

Notebook reproducible que orquesta el pipeline y reimprime las tablas y figuras
clave del informe a partir de los JSON de resultados en disco.

**El pipeline tiene 20 etapas en 9 pasos** (`paso1_datos` … `paso9_informe`):

1. `paso1_datos` — descarga Sen1Floods11 (7.5 GB), mide la normalización, define el lector y la pérdida enmascarada.
2. `paso2_modelo` — la U-Net propia (31 M de parámetros) y sus variantes de ablación.
3. `paso3_entrenamiento` — entrena, evalúa en distribución (entregable 3) y corre las 7 corridas de ablación (entregable 2).
4. `paso4_generalizacion` — U-Net contra Otsu con bootstrap pareado sobre chips (entregable 4).
5. `paso5_lima` — construye la región de Lima desde Copernicus EMS, baja la escena Sentinel-1 (ASF HyP3) y mide la brecha.
6. `paso6_figuras` — vistas PNG de los chips.
7. `paso7_fallos_sar` — separa los fallos del sensor de los del modelo (entregable 5).
8. `paso8_fusion_s2` — fusión con Sentinel-2, ganancia separada por nubosidad (entregable 6, bono).
9. `paso9_informe` — compila el informe NeurIPS en PDF.


## El comando único

Todo el circuito se corre con un solo comando (lo exige el enunciado):

```bash
python -m venv .venv
.venv\\Scripts\\activate            # Linux/Mac: source .venv/bin/activate
pip install -r pregunta2/requirements.txt
python pregunta2/ejecutar_todo.py
```

Cada etapa se salta si su artefacto ya existe, así que volver a lanzarlo hace
solo lo que falta. Variantes útiles:

```bash
python pregunta2/ejecutar_todo.py --listar     # qué hay hecho y qué falta
python pregunta2/ejecutar_todo.py --rapido     # el circuito entero en minutos (números NO son los del informe)
python pregunta2/ejecutar_todo.py --sin-bono   # omitir el entregable 6
python pregunta2/ejecutar_todo.py --desde 7    # retomar desde un paso
```

**Dependencias externas gratuitas.** La ejecución COMPLETA necesita descargar
Sen1Floods11 (7.5 GB) y, para la región de Lima, un Earthdata Login en `.env`
(`TOKEN_HYP3`) que habilita bajar la escena Sentinel-1 vía ASF HyP3. Sin esos
datos, este notebook igual reimprime las tablas y figuras desde los JSON ya
calculados que vienen en el repo.


## Preparación: localizar la raíz de la pregunta y cargar utilidades

In [ ]:
import json
from pathlib import Path

# La raíz de la pregunta es la carpeta que contiene ejecutar_todo.py.
AQUI = Path.cwd()
P2 = next((p for p in [AQUI, AQUI / "pregunta2", *AQUI.parents]
           if (p / "ejecutar_todo.py").is_file()), AQUI)
print("raíz de la pregunta:", P2)

def cargar(ruta_relativa):
    ruta = P2 / ruta_relativa
    if not ruta.is_file():
        print(f"(falta {ruta_relativa}: corré ejecutar_todo.py para generarlo)")
        return None
    return json.loads(ruta.read_text(encoding="utf-8"))


## Invocar el pipeline

La celda siguiente muestra el estado de las 20 etapas con `--listar` (no corre
nada pesado). Descomentá la segunda línea para lanzar el circuito completo; la
ejecución real descarga el dataset y puede tardar horas.

In [ ]:
import subprocess, sys

# Estado de las etapas (rápido, no descarga nada):
subprocess.run([sys.executable, str(P2 / "ejecutar_todo.py"), "--listar"], cwd=P2.parent)

# Circuito completo (descarga Sen1Floods11, entrena, evalúa, compila el informe):
# subprocess.run([sys.executable, str(P2 / "ejecutar_todo.py")], cwd=P2.parent)


## Entregable 2 — ablación (7 corridas)

Misma partición y semilla, cambiando una decisión por vez. Agregar etiquetas
débiles (`a_debiles_*`) empeora dentro de distribución y mejora fuera: salen de
una umbralización de Otsu, así que entrenar con ellas empuja al modelo hacia el
método clásico.

In [ ]:
ablacion = cargar("paso3_entrenamiento/ablacion.json")
if ablacion:
    print(f"{'corrida':<18}{'factor':<13}{'chips':>6}{'épocas':>8}"
          f"{'IoU test':>10}{'IoU Bolivia':>13}")
    for r in ablacion:
        print(f"{r['corrida']:<18}{r['factor']:<13}{r['chips_entrenamiento']:>6}"
              f"{r['epocas']:>8}{r['iou_test']:>10.4f}{r['iou_bolivia']:>13.4f}")
    print(f"\nOtsu (sin entrenar): test {ablacion[0]['otsu_test']:.4f}, "
          f"Bolivia {ablacion[0]['otsu_bolivia']:.4f}")


## Entregable 4 — generalización: U-Net contra Otsu

Bootstrap pareado sobre chips, 10 000 remuestreos. El orden se invierte entre
prueba y Bolivia, y las dos diferencias son significativas.

In [ ]:
comp = cargar("paso4_generalizacion/comparacion.json")
if comp:
    print(f"{'conjunto':<14}{'chips':>6}{'U-Net':>9}{'Otsu':>9}{'Δ':>10}"
          f"{'IC95%':>22}{'signif.':>9}")
    for nombre, c in comp["conjuntos"].items():
        ic = f"[{c['ic95'][0]:+.3f}, {c['ic95'][1]:+.3f}]"
        print(f"{nombre:<14}{c['chips']:>6}{c['iou_unet']:>9.4f}{c['iou_otsu']:>9.4f}"
              f"{c['delta']:>+10.4f}{ic:>22}{str(c['significativa']):>9}")


## Entregable 6 (bono) — fusión con Sentinel-2

La misma U-Net entrenada tres veces (solo SAR, solo óptico, fusión). El óptico y
la fusión suben mucho la IoU, pero la fusión no le gana al óptico solo; el radar
solo vuelve a pesar bajo nube.

In [ ]:
fus = cargar("paso8_fusion_s2/fusion.json")
if fus:
    print("IoU de prueba por modo:")
    for modo, c in fus["corridas"].items():
        print(f"  {modo:<8}{c['canales']} canales   IoU {c['conjuntos']['test']['iou_agua']:.4f}")
    print("\nIoU por nubosidad:")
    print(f"  {'modo':<10}{'todos':>9}{'despejados':>13}{'nublados':>11}")
    for modo, d in fus["por_nubosidad"].items():
        print(f"  {modo:<10}{d['todos']:>9.4f}{d['despejados']:>13.4f}{d['nublados']:>11.4f}")


## Figuras del informe

Las figuras que acompañan al informe viven en `paso9_informe/informe/figuras/`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

FIGS = P2 / "paso9_informe" / "informe" / "figuras"
nombres = ["separabilidad_vv.png", "fallo_especular.png", "etiqueta_lima.png"]
disponibles = [n for n in nombres if (FIGS / n).is_file()]
if disponibles:
    fig, ejes = plt.subplots(len(disponibles), 1,
                             figsize=(9, 5 * len(disponibles)))
    if len(disponibles) == 1:
        ejes = [ejes]
    for eje, n in zip(ejes, disponibles):
        eje.imshow(mpimg.imread(FIGS / n))
        eje.set_title(n)
        eje.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("(faltan las figuras: corré ejecutar_todo.py para generarlas)")


## Nota de reproducibilidad

Este notebook consolida; no vuelve a calcular nada. Para regenerar los JSON y
las figuras desde cero hace falta la ejecución completa de `ejecutar_todo.py`,
que descarga Sen1Floods11 (7.5 GB) y, para la región de Lima, la escena
Sentinel-1 vía ASF HyP3 (Earthdata Login en `.env`). Las tablas y figuras de
arriba son las mismas que lleva el informe NeurIPS en `paso9_informe/`.